In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import pickle

I0000 00:00:1781976349.319981     869 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781976349.634048     869 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781976350.631209     869 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=""'

# Add this before model.fit
import tensorflow as tf
tf.config.optimizer.set_jit(False)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=""'

SyntaxError: invalid syntax (349664209.py, line 7)

In [2]:
print(f"TF version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")
print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TF version: 2.21.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Built with CUDA: True
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
DATA_PATH = 'RML2016.10a_dict.pkl'

with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

In [6]:
all_keys = data.keys()
unique_mod_types = sorted(list(set([key[0] for key in all_keys])))
unique_snr_values = sorted(list(set([key[1] for key in all_keys])))

mods = unique_mod_types
snrs = unique_snr_values

X, Y, snr_labels = [], [], []

for mod_idx, mod_type in enumerate(mods):
    for snr_val in snrs:
        current_key = (mod_type, snr_val)
        if current_key in data:
            samples = data[current_key]          # shape: (1000, 2, 128)
            X.append(samples)
            Y.append(np.full(len(samples), mod_idx))
            snr_labels.append(np.full(len(samples), snr_val))

X          = np.vstack(X)               # (220000, 2, 128)
Y          = np.concatenate(Y)          # (220000,)
snr_labels = np.concatenate(snr_labels) # (220000,)

Y_cat = to_categorical(Y, num_classes=len(mods))  # (220000, 11)

X_train, X_test, Y_train, Y_test, snr_train, snr_test = train_test_split(
    X, Y_cat, snr_labels, test_size=0.2, random_state=42
)

# Transpose for Conv1D: (N, 2, 128) → (N, 128, 2)
X_train = X_train.transpose(0, 2, 1)
X_test  = X_test.transpose(0, 2, 1)

print(f"X_train: {X_train.shape}")  # expect (176000, 128, 2)
print(f"X_test:  {X_test.shape}")   # expect (44000, 128, 2)
print(f"Mods:    {mods}")

X_train: (176000, 128, 2)
X_test:  (44000, 128, 2)
Mods:    ['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']


In [7]:
def build_branch(inp):
    x = layers.Conv1D(64, 3, padding = 'same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, padding = "same", activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, padding = "same", activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    return x
raw_input = Input(shape = (128,2), name = 'raw_iq')

b1 = build_branch(raw_input)

def fft_features(x):
  iq = tf.cast(x, tf.float32)
  cplx = tf.complex(iq[..., 0], iq[..., 1])
  fft = tf.signal.fft(cplx)
  mag = tf.abs(fft)
  phase  = tf.math.angle(fft)
  return tf.stack([mag, phase], axis = -1)

fft_inp = layers.Lambda(fft_features, output_shape=(128, 2), name = 'fft_branch')(raw_input)
b2 = build_branch(fft_inp)

def envelope_features(x):
  iq = tf.cast(x,tf.float32)
  I,Q = iq[...,0], iq[...,1]
  env = tf.sqrt(I**2 + Q**2 + 1e-8)
  phase = tf.math.atan2(Q,I)
  return tf.stack([env, phase], axis=-1)

env_inp = layers.Lambda(envelope_features, output_shape=(128, 2), name = 'env_branch')(raw_input)
b3 = build_branch(env_inp)

merged = layers.concatenate([b1, b2, b3])
x = layers.Dense(256, activation = 'relu')(merged)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation = 'relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
out = layers.Dense(11, activation = 'softmax', name = 'output')(x)

model = Model(inputs = raw_input, outputs = out, name = "ensemble_multibranchcnn")
model.summary()

I0000 00:00:1781976474.301743     869 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4134 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
W0000 00:00:1781976474.543587    1049 nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_launcher.runfiles/nvidia_nvshmem
  ipykernel_launcher.runfiles/cuda_nvvm
  ipykernel_launcher.runfiles/cuda_cudart
  /usr/local/cuda
  /opt/cuda
  /var/data/python/lib/python3.13/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc
  /var/data/python/lib/python3.13/site-packages/tensorflow/python/platform/../../../../nvidia/cuda_nvcc
  /var/

Model: "ensemble_multibranchcnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ raw_iq (InputLayer) │ (None, 128, 2)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_branch (Lambda) │ (None, 128, 2)    │          0 │ raw_iq[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ env_branch (Lambda) │ (None, 128, 2)    │          0 │ raw_iq[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 64)   │        448 │ raw_iq[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 128, 64)   │        448 │ fft_branch[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 128, 64)   │        448 │ env_branch[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 128)  │     24,704 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 128, 128)  │     24,704 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 128, 128)  │     24,704 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_7[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 128, 128)  │     49,280 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 128, 128)  │     49,280 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 128, 128)  │     49,280 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_8[0][0]  

 Total params: 361,547 (1.38 MB)

 Trainable params: 358,859 (1.37 MB)

 Non-trainable params: 2,688 (10.50 KB)

In [9]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience = 10, restore_best_weights = True, monitor = 'val_accuracy'),
    ReduceLROnPlateau(factor = 0.5, patience=5, monitor = 'val_loss', min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model.keras',
                    save_best_only = True, monitor='val_accuracy', verbose =1)
]

history = model.fit(
    X_train, Y_train,validation_split=0.1,
    batch_size = 1024,
    epochs = 100,
    callbacks = callbacks,
    verbose = 1
)

Epoch 1/100


I0000 00:00:1781976528.524537    1041 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_19543__.94
W0000 00:00:1781976528.709957    1041 op_kernel.cc:1858] OP_REQUIRES failed at xla_ops.cc:602 : INTERNAL: Autotuner could not compile any configs for HLO: %gemm_fusion_MatMul.16 = f32[1024,11]{1,0} fusion(%SelectV2.50, %arg73.1), kind=kCustom, calls=%gemm_fusion_MatMul.16_computation, frontend_attributes={grad_x="false",grad_y="false"}, metadata={op_type="MatMul" op_name="ensemble_multibranchcnn_1/output_1/MatMul" source_file="/var/data/python/lib/python3.13/site-packages/tensorflow/python/framework/ops.py" source_line=1221}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_gemm"},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
W0000 00:00:1781976528.711033    1041 local_rendezvous.cc:412] Local rendezvous is aborting with status: INTERNAL: Autotuner could 

InternalError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/h/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/h/anaconda3/lib/python3.13/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/h/anaconda3/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/h/anaconda3/lib/python3.13/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/h/anaconda3/lib/python3.13/asyncio/base_events.py", line 683, in run_forever

  File "/home/h/anaconda3/lib/python3.13/asyncio/base_events.py", line 2050, in _run_once

  File "/home/h/anaconda3/lib/python3.13/asyncio/events.py", line 89, in _run

  File "/home/h/anaconda3/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 519

In [ ]:
custom_objs = {
    'fft_features': fft_features,
    'envelope_features': envelope_features
}

model = tf.keras.models.load_model('/content/drive/MyDrive/collab/best_model.keras', custom_objects=custom_objs)

snr_acc = {}
for snr in sorted(set(snr_test)):
    mask = snr_test == snr
    preds = model.predict(X_test[mask], verbose=0)
    correct = np.argmax(preds, axis=1) == np.argmax(Y_test[mask], axis=1)
    snr_acc[snr] = correct.mean()

plt.figure(figsize=(10, 5))
plt.plot(list(snr_acc.keys()), list(snr_acc.values()), marker='o', linewidth=2, color='steelblue')
plt.axhline(1/11, color='gray', linestyle='--', label='Random baseline (1/11)')
plt.axhline(0.6324, color='red', linestyle='--', label='SOTA 63.24%')
plt.xlabel('SNR (dB)')
plt.ylabel('Accuracy')
plt.title('Per-SNR Classification Accuracy — MultibranchCNN')
plt.xticks(list(snr_acc.keys()))
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/collab/snr_accuracy.png', dpi=150)
plt.show()